# Човек у петљи: Прелиминарни филтери, рангирање ризика и евиденција ревизије

README за ову лекцију уводи Човека у петљи кроз кратки сегмент који кориснику поставља питање да ли да `ОДОБРИ` или `ОДБИЈЕ` након што агент већ произведе одговор. Тај образац је добар почетак, али производне имплементације Човека у петљи обично захтевају још три додатне функције:

1. **Прелиминарни филтер** који се покреће **пре** него што агент изврши ризичан корак, како би се контролисали трошкови, неотуђивост и кашњење.
2. **Рангирање ризика**, тако да се радње са ниским ризиком аутоматски извршавају, радње са средњим ризиком се групно одобравају, а само радње великиог ризика захтевају људску интервенцију.
3. **Евиденција ревизије плус петља за ревизију**, тако да свака одлука филтера буде забележена као JSONL, а одбијање поново покреће агента са структуираним разлогом уместо само да пише `Преиспитивање...`.

Овај бележник гради све ове функције вршећи их на истим примитивима као у `06-system-message-framework.ipynb`. Извршава се од почетка до краја у режиму `DEMO_MODE = True` (није потребна интерактивна улазна тачка) или уз стварне `input()` упите када је `DEMO_MODE = False`. Напомена: у DEMO_MODE трећи циљни покушај је скриптован тако да су механизми петље видљиви од почетка до краја. Стварна ревизијом подржана рекласификација захтева `DEMO_MODE = False` и оператера.

**Изван обима ове лекције (покривено у другим лекцијама):** аутентификација и контрола приступа (лекција 06 README претња 2), middleware за позив алата (лекција 14 MAF дубински преглед), модели дебате са више агената.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: Прелаз пред акцијом

README-ов HITL фрагмент прво позива агента, а затим тражи од корисника да одобри излаз. То је **пост-акцијски** ток. Агент је већ извршио радњу, тако да је трошак позива LLM-а већ плаћен, и било који споредни ефекат (послат емаил, уписан ред у базу података, објављен коментар) је већ настало.

**Прелаз пред акцијом** убацује прелаз пре него што агент изведе ризичан корак. Агент предлаже радњу, прелаз одлучује да ли ће се извршити, и само уз одобрење долази до споредног ефекта.

| Аспект | Одобрење након акције (фрагмент из README-а) | Прелаз пред акцијом (овај нотебук) |
|---|---|---|
| Када се извршава одобрење? | Након што је агент произвео излаз | Пре извршавања било каквог споредног ефекта |
| Трошак LLM-а при одбијању | Већ плаћен | Плаћен само за предлог, не за радњу |
| Неповратни споредни ефекти | Могући (радња је већ извршена) | Спречени |
| Јасноћа ревизије | Одобрење је излаз у конзоли | Одобрење је JSON запис са временском ознаком, радњом, разлогом |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Образац 2: Рангирање ризика

Ни свака акција не захтева људско одобрење. Преглед само за читање преко јавног API-ја има другачију тежину од слања имејла купцу. Поступање са оба на исти начин троши пажњу оператера и успорава агента.

Једноставан модел са 3 нивоа:

| Ниво | Примери | Ток одобрења |
|---|---|---|
| `низак` (само за читање) | Претраживање базе знања, проналажење опција лета, преузимање јавне веб странице | Аутоматско извршавање, евидентирано за ревизију |
| `средњи` (јефтина мутација) | Кеширање резултата, повећање бројача, заказивање подсетника | Аутоматско извршавање, али групно дневно прегледање |
| `висок` (спољашњи или неповратни) | Слање имејла, наплата картице, постављање на јавни канал | Блокирање док се не добије људско одобрење |

Ово је једно рангирање. Производни системи често користе прецизније нивое (нпр. AWS IAM нивоа дозвола, нивое приступа засноване на улогама). Верзија са 3 нивоа испод је најмања корисна верзија за агента који меша акције само за читање и оне са споредним ефектима.

Класификатор испод користи кључне речи као хеуристику како би демо остао детерминистички и јефтин. У производном систему бисте овај заменили наученим класификатором или полицијским мотором.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Образац 3: Аудит лог и петља ревизије

`print("Response approved.")` није аудит лог. За поверење, свака одлука капије треба да буде забележена као структуриран догађај који касније можете да упитујете, репродукујете или прикачите на преглед инцидента.

Два дела:

1. **JSONL само за додавање.** Једна линија по одлуци, са временском ознаком, акцијом, нивоом, одлуком, разлогом. Лако се претражује, лако се касније шаље у прави лог фајл.
2. **Петља ревизије при одбијању.** Када капија врати `deny`, агент поново покреће питање са разлогом одбијања у контексту, тако да следећи предлог може да избегне проблем.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Додатни ресурси

Више других јавних пројеката имплементира варијације ових HITL образаца. Упоредите приступе да бисте пронашли шта одговара вашем стеку:

- **LangChain** омотаци алата за човека у петљи ([документација](https://python.langchain.com/docs/integrations/tools/human_tools)): омотаци алата који заустављају извршавање за унос од човека.
- **AutoGen** `UserProxyAgent` ([v0.2 документација](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ је ово реструктурирао): користи улогу агента посебно да представља човека у разговорима више агената.
- **Semantic Kernel** филтери функција ([документација](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): посредник који се покреће око сваког позива функције, погодан за контролу логике.

Сваки пројекат другачије рукује три подобразаца: LangChain их омотава као алате, AutoGen користи улогу агента, Semantic Kernel користи посредничке филтере. Прочитајте једну или две имплементације у целини пре него што одаберете дизајн за свог агента.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
